# Experiment 002: Colab GPU Runner

This notebook is a thin, auditable launcher for the repository's Experiment 002 code. It compares the transparent Transformers baseline with vLLM on the same Colab GPU. The pilot validates the real-model pipeline; it is not a final performance claim.

Before running: choose **Runtime → Change runtime type → GPU**. Run every cell in order, and download the artifact before the runtime is reclaimed.

In [ ]:
# Record the assigned hardware before installing anything.
!nvidia-smi
import platform, sys
print('Python:', sys.version)
print('Platform:', platform.platform())

In [ ]:
# Clone once. Re-running this cell updates nothing silently.
import os, subprocess
repo = '/content/llm-systems-lab'
if not os.path.exists(repo):
    subprocess.run(['git', 'clone', 'https://github.com/bigporcupine/llm-systems-lab.git', repo], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', commit)

In [ ]:
# Resolve the moving model name to an immutable revision.
%pip install -q huggingface_hub
import json
from pathlib import Path
from huggingface_hub import model_info
source_config = Path('experiments/exp002_serving_engines/pilot.json')
config = json.loads(source_config.read_text())
config['model_revision'] = model_info(config['model']).sha
runtime_config = Path('/content/exp002-pilot-resolved.json')
runtime_config.write_text(json.dumps(config, indent=2) + '\n')
print(runtime_config.read_text())

## Install isolated backend environments

The installations are intentionally separate so vLLM cannot silently replace the Transformers baseline's PyTorch or kernel dependencies. This can take several minutes.

In [ ]:
!bash scripts/colab/setup_backend.sh transformers
!bash scripts/colab/setup_backend.sh vllm

In [ ]:
# Helpers start one backend, wait for readiness, run the shared matrix, and stop it.
import json, os, signal, subprocess, time
from pathlib import Path
config = json.loads(Path('/content/exp002-pilot-resolved.json').read_text())
results_root = Path('/content/exp002-results')
results_root.mkdir(exist_ok=True)

def run_backend(backend, port):
    log_path = results_root / f'{backend}-server.log'
    log_handle = log_path.open('w')
    process = subprocess.Popen([
        'bash', 'scripts/colab/start_backend.sh', backend,
        config['model'], config['model_revision'], config['dtype'], str(port),
    ], stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)
    try:
        subprocess.run([
            sys.executable, 'scripts/colab/wait_for_server.py',
            '--url', f'http://127.0.0.1:{port}/health',
            '--timeout-seconds', '900',
        ], check=True)
        benchmark_python = '.venv-transformers/bin/python'
        backend_python = f'.venv-{backend}/bin/python'
        package_name = 'transformers' if backend == 'transformers' else 'vllm'
        backend_version = subprocess.check_output([
            backend_python, '-c',
            f'import importlib.metadata; print(importlib.metadata.version(\"{package_name}\"))',
        ], text=True).strip()
        subprocess.run([
            benchmark_python, '-m', 'llm_systems_lab', 'experiment-002-backend',
            '--backend', backend,
            '--base-url', f'http://127.0.0.1:{port}/v1',
            '--config', '/content/exp002-pilot-resolved.json',
            '--output-dir', str(results_root / backend),
            '--backend-version', backend_version,
        ], check=True)
    finally:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
        log_handle.close()
        time.sleep(10)
        subprocess.run(['nvidia-smi'])

In [ ]:
run_backend('transformers', 8000)

In [ ]:
run_backend('vllm', 8001)

In [ ]:
# Package immediately: Colab runtime storage is ephemeral.
import shutil
archive = shutil.make_archive('/content/exp002-results', 'zip', '/content/exp002-results')
print('Created:', archive)
from google.colab import files
files.download(archive)